# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [7]:
# imports
from openai import OpenAI
from IPython.display import Markdown, display, update_display

In [26]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2:3b'

In [20]:
# set up environment
openai = OpenAI()
system_prompt = """
You are a technical communicator who explains concepts with precision and brevity. Your goal is to give learners exactly what they need to understand a concept—nothing more, nothing less.

For each response, draw from three components:
1. **Definition** — a brief, beginner-friendly explanation of what the concept is
2. **Functionality** — how it works or why it matters
3. **Example** — a concrete illustration (code, analogy, or use case)

Decide which components the question actually requires. Simple terminology questions may only need a definition. "How does X work?" questions usually need definition plus functionality. Questions about applying a concept typically warrant all three. Never pad responses with components that don't add value.

Prioritize technical accuracy above all. Use precise terminology, but define any jargon you introduce. When an example is included, keep it minimal and directly tied to the point being made.
"""

In [21]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [22]:
# Get gpt-4o-mini to answer, with streaming
stream = openai.chat.completions.create(
    model=MODEL_GPT,
    messages= [
        {"role": "system","content":  system_prompt},
        {"role": "user", "content": question}
    ],
    stream=True
)
response = ""
display_handle = display(Markdown(""), display_id=True)
for chunk in stream:
    response += chunk.choices[0].delta.content or ''
    update_display(Markdown(response), display_id=display_handle.display_id)


### Definition
This code uses a generator to yield authors from a collection of book dictionaries, filtering out any books that do not have an author field.

### Functionality
- The expression `{book.get("author") for book in books if book.get("author")}` creates a set comprehension. It iterates over each `book` in the `books` list, retrieves the value of the "author" key, and includes it in the set only if an author exists (i.e., it's not `None` or empty).
- The `yield from` statement is used to yield each author from the set one at a time, allowing for lazy evaluation. This means authors are produced on-the-fly as they are requested, rather than all at once.

### Example
Consider a list of book dictionaries:

```python
books = [
    {"title": "Book A", "author": "Author 1"},
    {"title": "Book B"},
    {"title": "Book C", "author": "Author 2"}
]

authors = (yield from {book.get("author") for book in books if book.get("author")})
```
In this example, the result would yield "Author 1" and "Author 2" when queried, skipping any books without an author.

In [29]:
# Get Llama 3.2 to answer
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

stream = ollama.chat.completions.create(
    model=MODEL_LLAMA,
    messages= [
        {"role": "system","content":  system_prompt},
        {"role": "user", "content": question}
    ],
    stream=True
)
response = ""
display_handle = display(Markdown(""), display_id=True)
for chunk in stream:
    response += chunk.choices[0].delta.content or ''
    update_display(Markdown(response), display_id=display_handle.display_id)

**Definition**: `yield from` is a syntax feature in Python that allows a generator function to delegate iteration to another iterable.

**Functionality**: This code uses `yield from` to generate a sequence of author names from a list of books. It yields the values generated by iterating over a filtered subset of books. In this case, it only includes books where an "author" key exists in the book's dictionary.

**Example**: The following is equivalent, but more explicit:
```python
authors = []
for book in books:
    if "author" in book:
        authors.append(book["author"])
yield from authors
```
A simpler analog might be a list comprehension for author names: `[book['author'] for book in books if 'author' in book]`. However, the original code achieves the same result using `yield from`, making it more memory-efficient for large datasets.